# Figure Generation Notebook

Generates the figures referenced by [knowledge.md](../knowledge.md). All
output PNGs are saved next to this notebook under `../figures/`.

**Structure**

- **Common setup** — helpers used by all sections.
- **Section 1: RoPE** — five figures illustrating 2D rotation and RoPE.
- *(future sections go below; copy the pattern)*

To regenerate everything, just **Run All**.

## Common setup

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

# All figures land in docs/figures/, regardless of where the kernel was started.
FIG_DIR = (Path.cwd() / "../figures").resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def setup_axes(ax, lim=5.5, title=""):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(title, fontsize=11)


def arrow(ax, v, color="blue", label=None, lw=2):
    ax.annotate(
        "",
        xy=(v[0], v[1]),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color=color, lw=lw),
    )
    if label:
        ax.text(v[0] * 1.15, v[1] * 1.15, label, color=color, fontsize=10, fontweight="bold")


def save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=110, bbox_inches="tight")
    print(f"saved {path.relative_to(FIG_DIR.parent.parent)}")

## Section 1: RoPE

Five figures supporting §21 of [knowledge.md](../knowledge.md).

### fig1 — one vector rotated through many angles

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
setup_axes(ax, 5.5, "One vector rotated through many angles (tip traces a circle)")

v0 = np.array([3.0, 4.0])
r = 5.0
t = np.linspace(0, 2 * np.pi, 200)
ax.plot(r * np.cos(t), r * np.sin(t), "k--", alpha=0.3)

rotations = [0, 30, 60, 90, 135, 180, 270]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(rotations)))
for deg, color in zip(rotations, colors):
    th = np.radians(deg)
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    arrow(ax, R @ v0, color=color, label=f"+{deg} deg")

plt.tight_layout()
save(fig, "fig1_rotation_circle.png")
plt.show()

### fig2 — basis vectors before and after rotation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
theta_deg = 35
th = np.radians(theta_deg)

setup_axes(axes[0], 1.5, "Before: basis vectors x and y")
arrow(axes[0], [1, 0], "red", "x = (1,0)")
arrow(axes[0], [0, 1], "green", "y = (0,1)")

setup_axes(axes[1], 1.5, f"After rotation by theta = {theta_deg} deg")
arrow(axes[1], [1, 0], "red", lw=1)
arrow(axes[1], [0, 1], "green", lw=1)
x_rot = [np.cos(th), np.sin(th)]
y_rot = [-np.sin(th), np.cos(th)]
arrow(axes[1], x_rot, "red", f"x = ({x_rot[0]:.2f},{x_rot[1]:.2f})")
arrow(axes[1], y_rot, "green", f"y = ({y_rot[0]:.2f},{y_rot[1]:.2f})")
axes[1].add_patch(Arc((0, 0), 0.5, 0.5, theta1=0, theta2=theta_deg, color="red"))

plt.tight_layout()
save(fig, "fig2_basis.png")
plt.show()

### fig3 — RoPE clock-hand view (8 positions, 30 deg/step)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
axes = axes.flatten()
theta_per = np.radians(30)
v0 = np.array([1.0, 0.0])
t = np.linspace(0, 2 * np.pi, 100)

for i, ax in enumerate(axes):
    angle = i * theta_per
    R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    setup_axes(ax, 1.4, f"Position i={i}, angle={np.degrees(angle):.0f} deg")
    ax.plot(np.cos(t), np.sin(t), "k--", alpha=0.2)
    arrow(ax, R @ v0, "blue", lw=3)
    if angle > 0:
        ax.add_patch(Arc((0, 0), 0.6, 0.6, theta1=0, theta2=np.degrees(angle), color="red"))

plt.suptitle("RoPE clock-hand view: rotates by 30 deg per position step", fontsize=13, y=1.0)
plt.tight_layout()
save(fig, "fig3_positions.png")
plt.show()

### fig4 — multi-scale frequencies (4 pairs at very different speeds)

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(17, 13))
positions = [0, 1, 5, 25, 100]
freqs = [60, 20, 6, 2]
labels = [
    "Pair 0\n(fastest)\n60 deg/step",
    "Pair 1\n20 deg/step",
    "Pair 2\n6 deg/step",
    "Pair 3\n(slowest)\n2 deg/step",
]
colors = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]
t = np.linspace(0, 2 * np.pi, 100)

for row, (fr, lab) in enumerate(zip(freqs, labels)):
    for col, pos in enumerate(positions):
        ax = axes[row, col]
        a = np.radians(pos * fr)
        v = np.array([np.cos(a), np.sin(a)])
        setup_axes(ax, 1.4, f"pos={pos}, angle={pos*fr} = {(pos*fr) % 360} deg")
        ax.plot(np.cos(t), np.sin(t), "k--", alpha=0.2)
        arrow(ax, v, colors[row], lw=3)
        if col == 0:
            ax.text(-2.8, 0, lab, fontsize=11, fontweight="bold", va="center", ha="center")

plt.suptitle("Multi-scale RoPE: each pair rotates at its own speed", fontsize=14, y=1.0)
plt.tight_layout()
save(fig, "fig4_multi_freq.png")
plt.show()

### fig5 — same gap → same dot product (relative position)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

q = np.array([0.8, 0.2])
k = np.array([0.6, 0.6])
scenarios = [(0, 2), (3, 5), (10, 12)]
theta = np.radians(30)

for ax, (i, j) in zip(axes, scenarios):
    setup_axes(ax, 1.3, f"q at pos i={i}, k at pos j={j} (gap = {j - i})")
    R_i = np.array([[np.cos(i * theta), -np.sin(i * theta)], [np.sin(i * theta), np.cos(i * theta)]])
    R_j = np.array([[np.cos(j * theta), -np.sin(j * theta)], [np.sin(j * theta), np.cos(j * theta)]])
    qr = R_i @ q
    kr = R_j @ k
    arrow(ax, qr, "blue", f"q rotated {i * 30} deg")
    arrow(ax, kr, "red", f"k rotated {j * 30} deg")
    dot = float(np.dot(qr, kr))
    ax.text(
        0,
        -1.18,
        f"q dot k = {dot:.4f}",
        fontsize=12,
        ha="center",
        bbox=dict(boxstyle="round", facecolor="yellow", alpha=0.9),
    )

plt.suptitle("Same gap (j-i=2) -> identical dot product, all 3 scenarios", fontsize=14, y=1.02)
plt.tight_layout()
save(fig, "fig5_relative.png")
plt.show()

## Adding a new section

Copy this template below to add another figure group (e.g. attention masks,
SwiGLU activation curves). Keep figure filenames descriptive and prefix
them with the section topic so they sort naturally in `docs/figures/`.

```python
fig, ax = plt.subplots()
# ... plot ...
save(fig, "<topic>_<name>.png")
plt.show()
```

# RoPE 2D Rotation Visualizations

Run each cell to see proper geometric plots of what rotation actually does.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Arc

def setup_axes(ax, lim=5.5, title=''):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_title(title)

def arrow(ax, v, color='blue', label=None, lw=2):
    ax.annotate('', xy=(v[0], v[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    if label:
        ax.text(v[0]*1.1, v[1]*1.1, label, color=color, fontsize=12, fontweight='bold')

## 1. Basic 2D rotation — one vector at several angles

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
setup_axes(ax, lim=5.5, title='One vector rotated through several angles\n(tip traces a circle)')

v0 = np.array([3, 4])  # length 5, starting at ~53°
r = np.linalg.norm(v0)
phi0 = np.arctan2(v0[1], v0[0])

# Draw the circle of radius r
theta_circle = np.linspace(0, 2*np.pi, 200)
ax.plot(r*np.cos(theta_circle), r*np.sin(theta_circle), 'k--', alpha=0.3, label=f'circle of radius {r}')

# Draw the vector at several rotation angles
rotations = [0, 30, 60, 90, 135, 180, 270]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(rotations)))
for deg, color in zip(rotations, colors):
    theta = np.radians(deg)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    v_rot = R @ v0
    arrow(ax, v_rot, color=color, label=f'+{deg}°')

ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('rope_fig1_basic_rotation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Note: all arrows have the same length (5). Only the angle changes.')

## 2. The basis vectors rotated (where the rotation matrix comes from)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
theta_deg = 35
theta = np.radians(theta_deg)

# Before
ax = axes[0]
setup_axes(ax, lim=1.5, title='Before rotation: basis vectors x̂ and ŷ')
arrow(ax, [1, 0], color='red',   label='x̂ = (1, 0)')
arrow(ax, [0, 1], color='green', label='ŷ = (0, 1)')

# After
ax = axes[1]
setup_axes(ax, lim=1.5, title=f'After rotation by θ = {theta_deg}°')
# Draw ghosts of original basis vectors
arrow(ax, [1, 0], color='red',   lw=1)
arrow(ax, [0, 1], color='green', lw=1)
# Draw rotated
x_rot = [np.cos(theta), np.sin(theta)]
y_rot = [-np.sin(theta), np.cos(theta)]
arrow(ax, x_rot, color='red',   label=f'x̂\' = (cos θ, sin θ)\n   = ({x_rot[0]:.2f}, {x_rot[1]:.2f})', lw=2)
arrow(ax, y_rot, color='green', label=f'ŷ\' = (-sin θ, cos θ)\n   = ({y_rot[0]:.2f}, {y_rot[1]:.2f})', lw=2)
# Arc showing the rotation angle
ax.add_patch(Arc((0,0), 0.5, 0.5, angle=0, theta1=0, theta2=theta_deg, color='red'))
ax.text(0.32, 0.1, f'θ', fontsize=14, color='red')

plt.tight_layout()
plt.savefig('rope_fig2_basis.png', dpi=120, bbox_inches='tight')
plt.show()
print('The rotation matrix R(θ) = [[cos θ, -sin θ], [sin θ, cos θ]]')
print('Its COLUMNS are exactly where x̂ and ŷ land after rotation.')

## 3. RoPE positions — clock hand at different sequence positions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

theta_per_position = np.radians(30)  # 30° per position
starting_v = np.array([1.0, 0.0])

for i, ax in enumerate(axes):
    angle = i * theta_per_position
    R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    v_rot = R @ starting_v
    setup_axes(ax, lim=1.5, title=f'Position i={i}\nangle = {np.degrees(angle):.0f}°')
    # Draw circle
    t = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(t), np.sin(t), 'k--', alpha=0.2)
    # Draw the rotated vector
    arrow(ax, v_rot, color='blue', lw=3)
    # Draw arc for the angle
    if angle > 0:
        ax.add_patch(Arc((0,0), 0.6, 0.6, theta1=0, theta2=np.degrees(angle), color='red'))

plt.suptitle('RoPE clock-hand view: vector rotates by θ per position step', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('rope_fig3_clock_positions.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Multi-frequency clocks (the RoPE multi-scale idea)

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(18, 14))

positions = [0, 1, 5, 25, 100]
freqs_deg = [60, 20, 6, 2]  # 4 clocks at different speeds (deg per position)
freq_labels = ['Pair 0 (fastest): 60°/step',
               'Pair 1: 20°/step',
               'Pair 2: 6°/step',
               'Pair 3 (slowest): 2°/step']

for row, (freq_deg, freq_label) in enumerate(zip(freqs_deg, freq_labels)):
    for col, pos in enumerate(positions):
        ax = axes[row, col]
        angle = np.radians(pos * freq_deg)
        v = np.array([np.cos(angle), np.sin(angle)])
        setup_axes(ax, lim=1.4, title=f'pos={pos}, angle={pos*freq_deg}° ≡ {(pos*freq_deg)%360}°')
        t = np.linspace(0, 2*np.pi, 100)
        ax.plot(np.cos(t), np.sin(t), 'k--', alpha=0.2)
        arrow(ax, v, color=['#d62728','#ff7f0e','#2ca02c','#1f77b4'][row], lw=3)
        if col == 0:
            ax.text(-2.5, 0, freq_label, fontsize=11, fontweight='bold',
                    rotation=90, va='center', ha='center')

plt.suptitle('Multi-scale RoPE: each pair rotates at its own speed',
             fontsize=15, y=1.00)
plt.tight_layout()
plt.savefig('rope_fig4_multi_freq.png', dpi=120, bbox_inches='tight')
plt.show()
print('Fast pairs distinguish nearby positions; slow pairs distinguish far positions.')

## 5. Why dot product depends only on RELATIVE position

Two vectors $q$ and $k$ rotated by different amounts. The angle BETWEEN them is what matters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original q and k
q = np.array([0.8, 0.2])
k = np.array([0.6, 0.6])

scenarios = [
    (0, 2, 'q at pos i=0, k at pos j=2 → gap = 2'),
    (3, 5, 'q at pos i=3, k at pos j=5 → gap = 2'),
    (10, 12, 'q at pos i=10, k at pos j=12 → gap = 2'),
]
theta = np.radians(30)  # 30° per position step

for ax, (i, j, title) in zip(axes, scenarios):
    setup_axes(ax, lim=1.3, title=title)
    R_i = np.array([[np.cos(i*theta), -np.sin(i*theta)], [np.sin(i*theta), np.cos(i*theta)]])
    R_j = np.array([[np.cos(j*theta), -np.sin(j*theta)], [np.sin(j*theta), np.cos(j*theta)]])
    q_rot = R_i @ q
    k_rot = R_j @ k
    arrow(ax, q_rot, color='blue',  label=f'q rotated by {i*30}°')
    arrow(ax, k_rot, color='red',   label=f'k rotated by {j*30}°')
    dot = np.dot(q_rot, k_rot)
    ax.text(0, -1.15, f'q·k = {dot:.4f}', fontsize=13, ha='center',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    # Arc between them
    ang_q = np.degrees(np.arctan2(q_rot[1], q_rot[0]))
    ang_k = np.degrees(np.arctan2(k_rot[1], k_rot[0]))
    ax.legend(loc='lower right', fontsize=9)

plt.suptitle('Same gap (j-i=2) → identical dot product across all 3 scenarios',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('rope_fig5_relative.png', dpi=120, bbox_inches='tight')
plt.show()
print('Notice: the absolute positions i, j change, but the dot product is identical')
print('because the GAP (j-i) is always 2.')